# 矩阵乘法分块计算实验

矩阵乘法是高性能计算和深度学习中的典型基础计算，适合用来观察数据布局、片上存储、Cube计算和多核并行之间的关系。本实验围绕矩阵乘法$C=A\times B$展开，在Ascend C环境中设置单核静态Tensor基线、单核Matmul API和多核Matmul API三种实验方案。单核静态Tensor基线由一个AI Core完成外部存储（Global Memory，GM）与内部存储（Local Memory，LM）之间的数据搬运，并在LM中完成乘加累加；单核Matmul API实验方案使用Matmul高阶API和Tiling组织Cube计算；多核Matmul API实验方案进一步把输出区域分配给多个AI Core并行处理。

本实验学习大纲如下：

1. 环境准备：创建实验目录并加载CANN环境；
2. 问题分析：说明输入输出、计算公式、数据布局、分块策略和实验参数；
3. 核函数开发：实现单核静态Tensor基线核函数和Matmul API Tiling核函数，并以单核、多核配置运行Matmul API Tiling核函数；
4. 正确性验证：准备Host输入和CPU参考结果，开发核函数调用代码，完成构建、运行、误差校验和性能分析；
5. 实验总结：比较单核静态Tensor基线、单核Matmul API和多核Matmul API三种实验方案。


---
## 1. 环境准备

首先创建实验所需目录，并尝试加载Ascend CANN环境变量。

目录划分如下：

- `Source/02.02/Code/include`：保存Host侧公共头文件。
- `Source/02.02/Code`：保存输入生成、CPU参考计算和数据转换代码。
- `Source/02.02/ascend_ops/op_kernel`：保存Device侧核函数代码。
- `Source/02.02/ascend_ops/host_launch`：保存Host侧核函数调用代码。
- `Source/02.02/scripts`：保存完整构建与运行脚本。
- `Source/02.02/results`：保存实验结果。

如果当前环境已安装CANN，下面的代码会加载`set_env.sh`。如果没有找到该文件，仍然可以继续生成代码，但完整编译和运行需要在已配置CANN的环境中完成。


In [ ]:
!mkdir -p Source/02.02/Code/include
!mkdir -p Source/02.02/scripts
!mkdir -p Source/02.02/results
!mkdir -p Source/02.02/ascend_ops/host_launch
!mkdir -p Source/02.02/ascend_ops/op_kernel

import os
import shlex
import shutil
import subprocess
from pathlib import Path

legacy_kernel = Path("Source/02.02/ascend_ops/op_kernel/static_tensor_matmul.cpp")
if legacy_kernel.exists():
    legacy_kernel.unlink()
    print("Removed legacy kernel file:", legacy_kernel)

def find_cann_root(env_script):
    for parent in [env_script.parent, *env_script.parents]:
        cmake_candidates = [
            parent / "tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
            parent / "x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake",
        ]
        if any(path.is_file() for path in cmake_candidates):
            return parent
    return None

candidate_scripts = []
for item in [
    os.environ.get("ASCEND_TOOLKIT_HOME"),
    os.environ.get("ASCEND_INSTALL_PATH"),
    "/usr/local/Ascend/ascend-toolkit/latest",
]:
    if item:
        candidate_scripts.append(Path(item) / "set_env.sh")

for root in [Path("/usr/local/Ascend"), Path("/home/ma-user/Ascend"),
             Path("/opt/Ascend"), Path("/opt/conda/Ascend")]:
    if root.exists():
        candidate_scripts.extend(sorted(root.glob("**/set_env.sh"), reverse=True))

set_env = None
install_root = None
for script in candidate_scripts:
    if not script.is_file():
        continue
    root = find_cann_root(script)
    if root is not None:
        set_env = script
        install_root = root
        break

if set_env is not None and shutil.which("bash"):
    command = f"source {shlex.quote(str(set_env))} && env"
    loaded_env = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in loaded_env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(install_root)
    os.environ["ASCEND_CANN_PACKAGE_PATH"] = str(install_root)
    print("Ascend environment loaded from:", set_env)
    print("ASCEND_INSTALL_PATH:", install_root)
else:
    print("Ascend set_env.sh was not found. Code generation can continue; build/run needs CANN.")
    print("Diagnostic command: !find /usr/local /home/ma-user /opt -name set_env.sh 2>/dev/null")

print("Experiment directory:", Path("Source/02.02").resolve())
print("cmake:", shutil.which("cmake") or "not found")
print("C++ compiler:", shutil.which("g++") or shutil.which("c++") or "not found")


---
## 2. 问题分析

### 2.1 输入、输出和数据布局

本实验实现矩阵乘法：

$$C_{i,j}=\sum_{p=0}^{K-1}A_{i,p}\times B_{p,j}$$

Device侧输入A和B均采用ND行优先布局：

- A形状为`[M,K]`，索引为`i*K+p`；
- B形状为`[K,N]`，索引为`p*N+j`；
- C形状为`[M,N]`，使用FP32保存结果。

Host先生成FP32随机数据，再量化为FP16后传入Device；CPU参考结果也读取同一批FP16量化数据，以保证误差比较公平。


### 2.2 分块参数与多核策略

Matmul高阶API会把每个核负责的矩阵区域继续切分为`baseM×baseN×baseK`的Cube计算块。本实验允许通过`-p/-q`设置`baseM/baseN`，例如`64×64`；单核和多核Matmul API实验方案使用相同的请求值。

> **为什么不允许设置baseK？**
>
> 本实验环境所用CANN版本的`SetFixSplit`接口不支持用户固定`baseK`，该参数必须传`-1`。其原因是K方向块大小同时受输入数据类型、ND/NZ转换、16元素对齐、L0A/L0B以及L1可用空间等条件约束，需要由Tiling算法联合计算。代码因此调用`SetFixSplit(baseM, baseN, -1)`，再在`GetTiling()`完成后用`GetBaseK()`读取实际结果。

这意味着表格中的`baseK`（例如256）是CANN计算结果，不是命令行输入值。`-k`仍然表示矩阵维度K，与`baseK`不是同一个概念。


### 2.3 Host公共参数：`matmul_config.hpp`

`MatmulConfig`只保存Host生成输入和计算CPU参考结果所需的参数。CPU模拟程序已经移除，因此这里不再保留可切换的数据类型、布局和CPU分块配置：

- `m/n/k`表示完整矩阵规模；
- `seed`固定随机输入，便于重复实验。

Matmul分块参数、核数和测量次数由NPU Host程序的`Options`统一管理。


In [ ]:
%%writefile Source/02.02/Code/include/matmul_config.hpp

#pragma once

#include <cstddef>

namespace stmatmul {

struct MatmulConfig {
    std::size_t m = 512;
    std::size_t n = 512;
    std::size_t k = 512;
    unsigned seed = 20260506u;
};

}  // namespace stmatmul


---
## 3. 核函数开发

Device侧包含两个核函数入口，对应三种实验方案：单核静态Tensor基线使用独立的基线核函数；单核Matmul API和多核Matmul API共用同一个Tiling核函数，并分别使用单核Tiling和多核Tiling运行。

### 3.1 单核静态Tensor基线核函数

baseline固定启动1个AI Core，不使用Matmul高阶API。为了让参与点积的元素在GM中连续存放，Host为该实验方案提供转置后的$B^T$。Kernel以固定容量的静态`LocalTensor`作为LM缓冲区，将A的一行片段和$B^T$的一行片段从GM搬入LM，然后通过`LocalTensor::GetValue`完成FP32标量累加。

baseline不做M/N方向的矩阵分块或跨输出复用。K方向分段只用于适配有限的LM容量，不属于输出分块优化。由于GM↔LM搬运要求32字节对齐，Kernel每次计算连续8个FP32结果，并从LM统一搬回GM；这8个元素只是最小写回单元。该实验方案的`baseM/baseN/baseK`显示为`-1/-1/-1`，表示没有Matmul Tiling配置。


In [ ]:
%%writefile Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_baseline.cpp
// Baseline path: one AI Core, explicit GM <-> LM movement, no Matmul API.
// A and transposed-B rows are copied to statically sized LocalTensor buffers.
// Eight FP32 outputs are packed only to satisfy the 32-byte GM write alignment;
// this path does not apply M/N output tiling or cross-output data reuse.

#include "kernel_operator.h"

using namespace AscendC;

extern "C" __global__ __aicore__ void static_tensor_matmul_baseline(
    GM_ADDR a,
    GM_ADDR bTransposed,
    GM_ADDR c,
    uint32_t m,
    uint32_t n,
    uint32_t k)
{
    InitSocState();

    if (GetBlockIdx() != 0) {
        return;
    }

    GlobalTensor<half> aGm;
    GlobalTensor<half> bTransposedGm;
    GlobalTensor<float> cGm;
    aGm.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(a),
                        static_cast<uint64_t>(m) * k);
    bTransposedGm.SetGlobalBuffer(reinterpret_cast<__gm__ half *>(bTransposed),
                                  static_cast<uint64_t>(n) * k);
    cGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(c),
                        static_cast<uint64_t>(m) * n);

    constexpr uint32_t kStaticKChunk = 1024;
    constexpr uint32_t kOutputPack = 8;  // 8 FP32 = 32 bytes
    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<half> aLocal = ubAllocator.Alloc<half, kStaticKChunk>();
    LocalTensor<half> bLocal = ubAllocator.Alloc<half, kStaticKChunk>();
    LocalTensor<float> cLocal = ubAllocator.Alloc<float, kOutputPack>();

    for (uint32_t i = 0; i < m; ++i) {
        for (uint32_t jBase = 0; jBase < n; jBase += kOutputPack) {
            for (uint32_t lane = 0; lane < kOutputPack; ++lane) {
                const uint32_t j = jBase + lane;
                float acc = 0.0f;
                for (uint32_t pBase = 0; pBase < k; pBase += kStaticKChunk) {
                    const uint32_t chunkLen =
                        ((k - pBase) < kStaticKChunk) ? (k - pBase) : kStaticKChunk;
                    DataCopy(aLocal,
                             aGm[static_cast<uint64_t>(i) * k + pBase],
                             chunkLen);
                    DataCopy(bLocal,
                             bTransposedGm[static_cast<uint64_t>(j) * k + pBase],
                             chunkLen);
                    PipeBarrier<PIPE_ALL>();

                    for (uint32_t p = 0; p < chunkLen; ++p) {
                        acc += static_cast<float>(aLocal.GetValue(p)) *
                               static_cast<float>(bLocal.GetValue(p));
                    }
                }
                cLocal.SetValue(lane, acc);
            }

            PipeBarrier<PIPE_ALL>();
            DataCopy(cGm[static_cast<uint64_t>(i) * n + jBase],
                     cLocal,
                     kOutputPack);
            PipeBarrier<PIPE_ALL>();
        }
    }
}


### 3.2 单核与多核Matmul API核函数

单核和多核Matmul API实验方案共用`static_tensor_matmul_tiling`入口。Host先通过`MultiCoreMatmulTiling`生成`TCubeTiling`，再上传到Device。

#### 3.2.1 复制Tiling数据

`CopyCubeTiling`将GM中的Tiling数据复制到当前AI Core可访问的局部结构。Tiling中不仅包含实际`baseM/baseN/baseK`，还包含`singleCoreM/singleCoreN`、实际核数和片上搬运参数。


In [ ]:
%%writefile Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
// Tiled paths: Ascend C Matmul high-level API on Cube cores.
// The host generates one single-core and one multi-core TCubeTiling.

#define ASCENDC_CUBE_ONLY
#include "kernel_operator.h"
#include "lib/matmul_intf.h"

using namespace AscendC;
using namespace matmul;

namespace {

__aicore__ inline void CopyCubeTiling(TCubeTiling *dst, GM_ADDR tilingGm)
{
    uint32_t *dst32 = reinterpret_cast<uint32_t *>(dst);
    auto src32 = reinterpret_cast<__gm__ uint32_t *>(tilingGm);
    for (uint32_t i = 0; i < sizeof(TCubeTiling) / sizeof(uint32_t); ++i) {
        dst32[i] = src32[i];
    }
}



#### 3.2.2 计算每个核负责的输出区域

`CalcMatmulOffsets`根据`GetBlockIdx()`和`singleCoreM/singleCoreN`计算当前核对应的A、B、C起始位置，并计算M、N方向的尾块大小。

偏移使用`int64_t`，避免较大矩阵下乘法结果溢出；尾块则通过`SetTail`交给Matmul对象处理。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
__aicore__ inline void CalcMatmulOffsets(uint32_t blockIdx,
                                         const TCubeTiling &tiling,
                                         int64_t &offsetA,
                                         int64_t &offsetB,
                                         int64_t &offsetC,
                                         int32_t &tailM,
                                         int32_t &tailN)
{
    const uint32_t mBlockCount = Ceil(tiling.M, tiling.singleCoreM);
    const uint32_t mBlockIdx = blockIdx % mBlockCount;
    const uint32_t nBlockIdx = blockIdx / mBlockCount;

    offsetA = static_cast<int64_t>(mBlockIdx) * tiling.singleCoreM * tiling.Ka;
    offsetB = static_cast<int64_t>(nBlockIdx) * tiling.singleCoreN;
    offsetC = static_cast<int64_t>(mBlockIdx) * tiling.singleCoreM * tiling.N +
              static_cast<int64_t>(nBlockIdx) * tiling.singleCoreN;

    const int32_t remainM =
        static_cast<int32_t>(tiling.M - mBlockIdx * tiling.singleCoreM);
    const int32_t remainN =
        static_cast<int32_t>(tiling.N - nBlockIdx * tiling.singleCoreN);
    tailM = remainM < static_cast<int32_t>(tiling.singleCoreM)
                ? remainM
                : static_cast<int32_t>(tiling.singleCoreM);
    tailN = remainN < static_cast<int32_t>(tiling.singleCoreN)
                ? remainN
                : static_cast<int32_t>(tiling.singleCoreN);
}

}  // namespace



#### 3.2.3 Matmul核函数入口与workspace

Kernel首先调用`InitSocState`并检查Tiling指针，再通过`GetSysWorkSpacePtr()`获取Matmul高阶API运行所需的系统workspace，并结合`TCubeTiling`注册Matmul对象。

`usedCoreNum`检查可以避免启动核数大于实际Tiling核数时，多余核心访问无效输出区域。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
extern "C" __global__ __aicore__ void static_tensor_matmul_tiling(
    GM_ADDR a,
    GM_ADDR b,
    GM_ADDR c,
    GM_ADDR workspace,
    GM_ADDR tilingGm)
{
    InitSocState();

    if (tilingGm == nullptr) {
        return;
    }

    using A_T = half;
    using B_T = half;
    using C_T = float;

    if (GetSysWorkSpacePtr() == nullptr) {
        return;
    }

    TPipe pipe;
    TCubeTiling tiling;
    CopyCubeTiling(&tiling, tilingGm);

    if (GetBlockIdx() >= tiling.usedCoreNum) {
        return;
    }



#### 3.2.4 GM Tensor绑定和尾块定位

A、B、C分别绑定为FP16、FP16和FP32的`GlobalTensor`。根据当前核的偏移取得局部GM视图，再把实际尾块M/N传给Matmul对象。

单核和多核Matmul API实验方案的数据类型、布局和Kernel代码完全相同，仅Host生成的单核/多核Tiling不同。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
    GlobalTensor<A_T> aGlobal;
    GlobalTensor<B_T> bGlobal;
    GlobalTensor<C_T> cGlobal;
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ A_T *>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ B_T *>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ C_T *>(c), tiling.M * tiling.N);

    int64_t offsetA = 0;
    int64_t offsetB = 0;
    int64_t offsetC = 0;
    int32_t tailM = 0;
    int32_t tailN = 0;
    CalcMatmulOffsets(GetBlockIdx(), tiling, offsetA, offsetB, offsetC, tailM, tailN);

    auto gmA = aGlobal[offsetA];
    auto gmB = bGlobal[offsetB];
    auto gmC = cGlobal[offsetC];



#### 3.2.5 注册Matmul对象并执行Cube计算

`REGIST_MATMUL_OBJ`把`TPipe`、系统workspace、Matmul对象和`TCubeTiling`关联起来。随后设置A、B，关闭bias，调用`IterateAll`完成数据搬运、Cube计算和C写回。

`baseK`不在kernel中硬编码。kernel完全按照Host生成并上传的`TCubeTiling`执行，因此不同CANN版本或芯片可能得到不同的实际`baseK`。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
    using AType = MatmulType<TPosition::GM, CubeFormat::ND, A_T>;
    using BType = MatmulType<TPosition::GM, CubeFormat::ND, B_T>;
    using CType = MatmulType<TPosition::GM, CubeFormat::ND, C_T>;
    using BiasType = MatmulType<TPosition::GM, CubeFormat::ND, C_T>;
    Matmul<AType, BType, CType, BiasType> mm;

    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling);
    mm.SetTensorA(gmA, false);
    mm.SetTensorB(gmB, false);
    mm.DisableBias();
    mm.SetTail(tailM, tailN);

    mm.IterateAll(gmC);
    mm.End();
}


---
## 4. 正确性验证

Host负责生成输入、计算CPU参考结果、初始化ACL、申请Device内存、生成Matmul Tiling、运行三种实验方案并验证结果。

三种实验方案共享同一批FP16输入和同一份FP32参考结果。baseline与Matmul API的累加顺序可能不同，因此需要同时观察最大绝对误差和最大相对误差，而不能要求结果逐位完全一致。

### 4.1 Host输入与CPU参考结果

本节在Host侧生成固定随机输入，将其量化为FP16，并构造单核静态Tensor基线所需的转置B。随后基于同一批量化输入计算CPU参考结果，用于校验三种NPU实验方案的正确性。CPU代码只负责输入准备和参考结果计算，不作为独立实验方案参与性能比较。

#### 4.1.1 输入数据结构：`data_utils.hpp`

`MatrixInput`保存三种实验方案共用的FP16输入，以及单核静态Tensor基线需要的转置B。`make_input`负责生成输入，`cpu_reference`负责计算Host侧参考结果。


In [ ]:
%%writefile Source/02.02/Code/include/data_utils.hpp

#pragma once

#include "matmul_config.hpp"

#include <cstddef>
#include <cstdint>
#include <vector>

namespace stmatmul {

struct MatrixInput {
    std::vector<uint16_t> a_fp16;
    std::vector<uint16_t> b_fp16;
    std::vector<uint16_t> b_transposed_fp16;
};

MatrixInput make_input(const MatmulConfig& cfg);
std::vector<float> cpu_reference(const MatrixInput& input, const MatmulConfig& cfg);

}  // namespace stmatmul


#### 4.1.2 `half_utils.hpp`

Host先使用float生成输入，再调用`float_to_half_bits`将输入转换为half位表示，作为Device核函数的输入。CPU参考计算会用`half_bits_to_float`读取同一批half输入，再用float完成累加。


In [ ]:
%%writefile Source/02.02/Code/include/half_utils.hpp

#pragma once

#include <cstdint>

namespace stmatmul {

uint16_t float_to_half_bits(float value) noexcept;
float half_bits_to_float(uint16_t value) noexcept;

}  // namespace stmatmul


#### 4.1.3 half输入转换：`half_utils.cpp`

`float_to_half_bits`用于把float输入量化为half位表示；`half_bits_to_float`用于把half位表示还原为float。两者共同保证Host输入准备、参考结果计算和Device输入之间的数据表示一致。


In [ ]:
%%writefile Source/02.02/Code/half_utils.cpp

#include "half_utils.hpp"

#include <cstring>

namespace stmatmul {

static inline uint32_t f32_to_bits(float value) noexcept {
    uint32_t bits = 0;
    std::memcpy(&bits, &value, sizeof(bits));
    return bits;
}

static inline float bits_to_f32(uint32_t bits) noexcept {
    float value = 0.0f;
    std::memcpy(&value, &bits, sizeof(value));
    return value;
}

uint16_t float_to_half_bits(float value) noexcept {
    const uint32_t x = f32_to_bits(value);
    const uint32_t sign = (x >> 16) & 0x8000u;
    const uint32_t raw_exponent = (x >> 23) & 0xffu;
    uint32_t mantissa = x & 0x007fffffu;
    int32_t exponent = static_cast<int32_t>(raw_exponent) - 127 + 15;

    if (raw_exponent == 0xffu) {
        if (mantissa == 0) {
            return static_cast<uint16_t>(sign | 0x7c00u);
        }
        return static_cast<uint16_t>(sign | 0x7c00u | (mantissa >> 13) | 0x0001u);
    }

    if (exponent <= 0) {
        if (exponent < -10) {
            return static_cast<uint16_t>(sign);
        }
        mantissa |= 0x00800000u;
        const uint32_t shift = static_cast<uint32_t>(14 - exponent);
        uint32_t half_mantissa = mantissa >> shift;
        const uint32_t round_bit = (mantissa >> (shift - 1)) & 1u;
        half_mantissa += round_bit;
        return static_cast<uint16_t>(sign | half_mantissa);
    }

    if (exponent >= 31) {
        return static_cast<uint16_t>(sign | 0x7c00u);
    }

    uint32_t half = sign | (static_cast<uint32_t>(exponent) << 10) | (mantissa >> 13);
    const uint32_t round = mantissa & 0x00001000u;
    if (round) {
        ++half;
    }
    return static_cast<uint16_t>(half);
}

float half_bits_to_float(uint16_t value) noexcept {
    const uint32_t sign = (static_cast<uint32_t>(value) & 0x8000u) << 16;
    int32_t exponent = static_cast<int32_t>((static_cast<uint32_t>(value) >> 10) & 0x1fu);
    uint32_t mantissa = static_cast<uint32_t>(value) & 0x03ffu;

    if (exponent == 0) {
        if (mantissa == 0) {
            return bits_to_f32(sign);
        }
        while ((mantissa & 0x0400u) == 0) {
            mantissa <<= 1;
            --exponent;
        }
        ++exponent;
        mantissa &= ~0x0400u;
    } else if (exponent == 31) {
        return bits_to_f32(sign | 0x7f800000u | (mantissa << 13));
    }

    const uint32_t exponent_bits = static_cast<uint32_t>(exponent + (127 - 15)) << 23;
    return bits_to_f32(sign | exponent_bits | (mantissa << 13));
}

}  // namespace stmatmul


#### 4.1.4 输入读取规则：`data_utils.cpp`

`data_utils.cpp`生成固定的FP16输入，并同时保存行优先B和转置后的$B^T$。CPU参考结果读取行优先B；转置后的$B^T$只提供给单核静态Tensor基线进行连续搬运。


In [ ]:
%%writefile Source/02.02/Code/data_utils.cpp

#include "data_utils.hpp"

#include "half_utils.hpp"

#include <random>

namespace stmatmul {



接下来生成固定随机输入。`make_input`先生成float随机值，再保存量化后的FP16输入以及B的辅助转置视图。Host会把`a_fp16`、行优先的`b_fp16`和`b_transposed_fp16`都上传到Device：baseline使用连续存放列数据的$B^T$，单核和多核Matmul API实验方案仍使用行优先B。三种实验方案表示的是同一组数学输入。


In [ ]:
%%writefile -a Source/02.02/Code/data_utils.cpp

MatrixInput make_input(const MatmulConfig& cfg) {
    MatrixInput input;
    input.a_fp16.resize(cfg.m * cfg.k);
    input.b_fp16.resize(cfg.k * cfg.n);
    input.b_transposed_fp16.resize(cfg.n * cfg.k);

    std::mt19937 rng(cfg.seed);
    std::uniform_real_distribution<float> dist(-1.0f, 1.0f);

    for (std::size_t i = 0; i < input.a_fp16.size(); ++i) {
        const float v = dist(rng) * 0.25f;
        input.a_fp16[i] = float_to_half_bits(v);
    }
    for (std::size_t i = 0; i < cfg.k; ++i) {
        for (std::size_t j = 0; j < cfg.n; ++j) {
            const float v = dist(rng) * 0.25f;
            input.b_fp16[i * cfg.n + j] = float_to_half_bits(v);
            input.b_transposed_fp16[j * cfg.k + i] = float_to_half_bits(v);
        }
    }
    return input;
}



最后计算CPU参考结果并检查Device输出。`cpu_reference`使用普通三重循环，固定读取FP16量化后的A和行优先B，并使用FP32完成累加。它只负责生成正确性参考结果，不是CPU模拟实验方案。


In [ ]:
%%writefile -a Source/02.02/Code/data_utils.cpp

std::vector<float> cpu_reference(const MatrixInput& input, const MatmulConfig& cfg) {
    std::vector<float> c(cfg.m * cfg.n, 0.0f);
    for (std::size_t i = 0; i < cfg.m; ++i) {
        const std::size_t c_row = i * cfg.n;
        for (std::size_t p = 0; p < cfg.k; ++p) {
            const float av = half_bits_to_float(input.a_fp16[i * cfg.k + p]);
            for (std::size_t j = 0; j < cfg.n; ++j) {
                const float bv = half_bits_to_float(input.b_fp16[p * cfg.n + j]);
                c[c_row + j] += av * bv;
            }
        }
    }
    return c;
}

}  // namespace stmatmul


### 4.2 Host侧核函数调用

Host侧代码负责三件关键工作：

1. 固定以1核启动单核静态Tensor基线，并为它上传转置后的B；
2. 为Matmul API生成请求相同`baseM/baseN`的单核与多核Tiling；
3. 记录CANN最终选择的`baseK`并统一输出正确性和性能指标。

#### 4.2.1 头文件、错误检查和参数结构

`MatmulLaunchInfo`保存序列化Tiling、实际`blockDim`、M/N方向核划分、实际`baseM/baseN/baseK`和workspace大小。所有ACL返回值均由`ACL_CHECK`检查。


In [ ]:
%%writefile Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
#include "data_utils.hpp"
#include "matmul_config.hpp"

#include <acl/acl.h>
#include "aclrtlaunch_static_tensor_matmul_baseline.h"
#include "aclrtlaunch_static_tensor_matmul_tiling.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"

#include <algorithm>
#include <cctype>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdlib>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

#ifndef SOC_VERSION
#define SOC_VERSION "ascend910b1"
#endif

#define ACL_CHECK(expr)                                                                            \
    do {                                                                                           \
        aclError _ret = (expr);                                                                    \
        if (_ret != ACL_SUCCESS) {                                                                 \
            throw std::runtime_error(std::string("[ACL ERROR] ") + #expr +                       \
                                     " failed, ret=" + std::to_string(_ret) +                    \
                                     " at " + __FILE__ + ":" + std::to_string(__LINE__));       \
        }                                                                                          \
    } while (0)

namespace {

struct Options {
    int device = 0;
    uint32_t m = 512;
    uint32_t n = 512;
    uint32_t k = 512;
    uint32_t blockM = 64;
    uint32_t blockN = 64;
    uint32_t baselineCores = 1;
    uint32_t optimizedCores = 8;
    int warmup = 2;
    int repeat = 5;
    unsigned seed = 20260506;
    std::string csv = "results/static_tensor_matmul_result.csv";
};

struct CaseResult {
    std::string kernel;
    std::string path;
    uint32_t m = 0, n = 0, k = 0;
    uint32_t cores = 0;
    int32_t baseM = -1, baseN = -1, baseK = -1;
    double avgUs = 0.0;
    double minUs = 0.0;
    double gflops = 0.0;
    double readGb = 0.0;
    double writeGb = 0.0;
    double maxAbs = 0.0;
    double maxRel = 0.0;
    bool pass = false;
};

struct MatmulLaunchInfo {
    std::vector<uint8_t> tilingBytes;
    uint32_t blockDim = 0;
    uint32_t mDim = 0;
    uint32_t nDim = 0;
    int32_t baseM = -1;
    int32_t baseN = -1;
    int32_t baseK = -1;
    size_t workspaceBytes = 0;
};

uint32_t parse_u32(const char *s, const char *name)
{
    unsigned long v = std::stoul(s);
    if (v == 0 || v > static_cast<unsigned long>(std::numeric_limits<uint32_t>::max())) {
        throw std::invalid_argument(std::string(name) + " must be positive and fit in uint32_t");
    }
    return static_cast<uint32_t>(v);
}



#### 4.2.2 参数解析与约束检查

NPU程序只接收`--block-m/--block-n`，不接收`--block-k`：

- `M/N`要求为16的倍数，K要求为64的倍数；
- `baseM/baseN`要求按16元素对齐；
- baseline和单核Matmul固定使用1核；
- 矩阵规模、分块和核数必须能够安全转换为`int32_t`。

这里的`--k`表示完整矩阵K维，不表示`baseK`。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
Options parse_args(int argc, char **argv)
{
    Options opt;
    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        auto need = [&](const char *name) -> const char * {
            if (i + 1 >= argc) throw std::invalid_argument(std::string("missing value after ") + name);
            return argv[++i];
        };
        if (arg == "--device") opt.device = std::stoi(need("--device"));
        else if (arg == "--m") opt.m = parse_u32(need("--m"), "m");
        else if (arg == "--n") opt.n = parse_u32(need("--n"), "n");
        else if (arg == "--k") opt.k = parse_u32(need("--k"), "k");
        else if (arg == "--block-m") opt.blockM = parse_u32(need("--block-m"), "block-m");
        else if (arg == "--block-n") opt.blockN = parse_u32(need("--block-n"), "block-n");
        else if (arg == "--baseline-cores") opt.baselineCores = parse_u32(need("--baseline-cores"), "baseline-cores");
        else if (arg == "--optimized-cores") opt.optimizedCores = parse_u32(need("--optimized-cores"), "optimized-cores");
        else if (arg == "--warmup") opt.warmup = std::stoi(need("--warmup"));
        else if (arg == "--repeat") opt.repeat = std::stoi(need("--repeat"));
        else if (arg == "--seed") opt.seed = static_cast<unsigned>(std::stoul(need("--seed")));
        else if (arg == "--csv") opt.csv = need("--csv");
        else if (arg == "--help" || arg == "-h") {
            std::cout << "Usage: static_tensor_matmul_ascend_demo [--device 0] [--m 512 --n 512 --k 512] "
                         "[--block-m 64 --block-n 64] [--optimized-cores 8]\n";
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    if (opt.warmup < 0 || opt.repeat <= 0) {
        throw std::invalid_argument("warmup must be >= 0 and repeat must be > 0");
    }
    if (opt.baselineCores != 1) {
        throw std::invalid_argument("the GM-LM baseline must use exactly one core");
    }
    if (opt.m % 16 != 0 || opt.n % 16 != 0 || opt.k % 64 != 0) {
        throw std::invalid_argument("M/N must be multiples of 16 and K must be a multiple of 64");
    }
    if (opt.blockM % 16 != 0 || opt.blockN % 16 != 0) {
        throw std::invalid_argument("Matmul baseM/baseN must be multiples of 16");
    }
    constexpr uint32_t kInt32Max = static_cast<uint32_t>(std::numeric_limits<int32_t>::max());
    if (opt.m > kInt32Max || opt.n > kInt32Max || opt.k > kInt32Max ||
        opt.blockM > kInt32Max || opt.blockN > kInt32Max ||
        opt.baselineCores > kInt32Max || opt.optimizedCores > kInt32Max) {
        throw std::invalid_argument("Matmul dimensions, base sizes and core count must fit in int32_t");
    }
    return opt;
}



#### 4.2.3 指标计算与Matmul Tiling生成

`gflops`、`check_result`和访问量估算用于统一评价三种实验方案。`make_matmul_launch_info`根据运行时SoC、矩阵形状、数据类型和请求核数创建`MultiCoreMatmulTiling`。

本节最重要的接口是：

```cpp
tilingApi.SetFixSplit(baseM, baseN, -1);
tilingApi.GetTiling(cubeTiling);
actualBaseK = tilingApi.GetBaseK();
```

`-1`明确表示不固定K方向块大小。Tiling算法结合L0A/L0B/L1容量和对齐约束选择实际`baseK`，所以程序只验证`baseM/baseN`，并把`GetBaseK()`结果写入表格和CSV。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
double gflops(uint32_t m, uint32_t n, uint32_t k, double us)
{
    if (us <= 0.0) return 0.0;
    return (2.0 * static_cast<double>(m) * n * k) / us / 1000.0;
}

void check_result(const std::vector<float> &out, const std::vector<float> &ref,
                  double &maxAbs, double &maxRel, bool &pass)
{
    constexpr double kAbsTol = 5e-2;
    constexpr double kRelTol = 5e-2;
    maxAbs = 0.0;
    maxRel = 0.0;
    pass = true;
    for (size_t i = 0; i < out.size(); ++i) {
        const double diff = std::abs(static_cast<double>(out[i]) - static_cast<double>(ref[i]));
        maxAbs = std::max(maxAbs, diff);
        const double denom = std::max(1e-7, std::abs(static_cast<double>(ref[i])));
        const double rel = diff / denom;
        maxRel = std::max(maxRel, rel);
        if (diff > kAbsTol && rel > kRelTol) {
            pass = false;
        }
    }
}

double estimate_baseline_read_gb(const Options &opt)
{
    const double readElems =
        2.0 * static_cast<double>(opt.m) * opt.n * opt.k;
    return readElems * sizeof(uint16_t) / 1e9;
}

double estimate_tiling_read_gb(const Options &opt, const MatmulLaunchInfo &launchInfo)
{
    const double halfBytes = sizeof(uint16_t);
    const double readElems =
        static_cast<double>(opt.m) * opt.k * launchInfo.nDim +
        static_cast<double>(opt.k) * opt.n * launchInfo.mDim;
    return readElems * halfBytes / 1e9;
}

std::string normalize_platform_soc_name(std::string name)
{
    constexpr char kPrefix[] = "ascend";
    if (name.size() < 6) return name;
    for (size_t i = 0; i < 6; ++i) {
        if (std::tolower(static_cast<unsigned char>(name[i])) != kPrefix[i]) {
            return name;
        }
    }

    std::string normalized = "Ascend";
    normalized.reserve(name.size());
    for (size_t i = 6; i < name.size(); ++i) {
        const unsigned char ch = static_cast<unsigned char>(name[i]);
        normalized.push_back(static_cast<char>(std::isalpha(ch) ? std::toupper(ch) : ch));
    }
    return normalized;
}

MatmulLaunchInfo make_matmul_launch_info(const Options &opt,
                                         const char *runtimeSocName,
                                         uint32_t requestedCores,
                                         uint32_t requestedBaseM,
                                         uint32_t requestedBaseN)
{
    const std::string requestedSoc =
        normalize_platform_soc_name(runtimeSocName != nullptr ? runtimeSocName : SOC_VERSION);
    auto platform = platform_ascendc::PlatformAscendCManager::GetInstance(requestedSoc.c_str());
    if (platform == nullptr) {
        platform = platform_ascendc::PlatformAscendCManager::GetInstance();
    }
    if (platform == nullptr) {
        throw std::runtime_error("failed to initialize AscendC platform for " + requestedSoc);
    }

    matmul_tiling::MultiCoreMatmulTiling tilingApi(*platform);
    if (tilingApi.SetDim(static_cast<int32_t>(requestedCores)) != 0 ||
        tilingApi.SetAType(matmul_tiling::TPosition::GM,
                           matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_FLOAT16,
                           false) != 0 ||
        tilingApi.SetBType(matmul_tiling::TPosition::GM,
                           matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_FLOAT16,
                           false) != 0 ||
        tilingApi.SetCType(matmul_tiling::TPosition::GM,
                           matmul_tiling::CubeFormat::ND,
                           matmul_tiling::DataType::DT_FLOAT) != 0 ||
        tilingApi.SetShape(static_cast<int32_t>(opt.m),
                           static_cast<int32_t>(opt.n),
                           static_cast<int32_t>(opt.k)) != 0 ||
        tilingApi.SetOrgShape(static_cast<int32_t>(opt.m),
                              static_cast<int32_t>(opt.n),
                              static_cast<int32_t>(opt.k)) != 0 ||
        tilingApi.EnableBias(false) != 0 ||
        tilingApi.SetBufferSpace(-1, -1, -1) != 0) {
        throw std::runtime_error("failed to configure MultiCoreMatmulTiling");
    }

    // CANN chooses baseK. Only baseM/baseN are fixed by this experiment.
    if (tilingApi.SetFixSplit(static_cast<int32_t>(requestedBaseM),
                              static_cast<int32_t>(requestedBaseN),
                              -1) != 0) {
        throw std::runtime_error("failed to configure Matmul baseM/baseN");
    }

    optiling::TCubeTiling cubeTiling{};
    if (tilingApi.GetTiling(cubeTiling) < 0) {
        throw std::runtime_error("failed to generate multi-core Matmul tiling");
    }

    int32_t blockDim = 0;
    int32_t mDim = 0;
    int32_t nDim = 0;
    if (tilingApi.GetCoreNum(blockDim, mDim, nDim) != 0 || blockDim <= 0) {
        throw std::runtime_error("failed to obtain Matmul launch blockDim");
    }

    MatmulLaunchInfo info;
    info.blockDim = static_cast<uint32_t>(blockDim);
    info.mDim = static_cast<uint32_t>(mDim);
    info.nDim = static_cast<uint32_t>(nDim);
    info.baseM = tilingApi.GetBaseM();
    info.baseN = tilingApi.GetBaseN();
    info.baseK = tilingApi.GetBaseK();
    info.workspaceBytes = platform->GetLibApiWorkSpaceSize();
    if (info.workspaceBytes == 0) {
        throw std::runtime_error("AscendC Matmul reported zero system workspace size");
    }

    const uint32_t tilingSize = cubeTiling.GetDataSize();
    if (tilingSize == 0) {
        throw std::runtime_error("AscendC Matmul generated empty tiling data");
    }
    info.tilingBytes.resize(tilingSize);
    cubeTiling.SaveToBuffer(info.tilingBytes.data(), tilingSize);
    return info;
}



#### 4.2.4 启动单核静态Tensor基线

`run_baseline`每次先清零C，再以固定1核启动`static_tensor_matmul_baseline`。传入的第二个矩阵是Host预先生成并上传的$B^T$，以便Kernel对A行和B列进行连续DMA搬运。同步stream后记录耗时，将结果复制回Host并校验。

该行结果名为`static_tensor_matmul_baseline`。`baseM/baseN/baseK`均为`-1`，表示baseline使用静态`LocalTensor`完成GM→LM→GM数据搬运，但不进行M/N方向的输出分块优化。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
CaseResult run_baseline(const Options &opt,
                        uint8_t *aDevice, uint8_t *bTransposedDevice, uint8_t *cDevice,
                        aclrtStream stream, const std::vector<float> &ref)
{
    const size_t cBytes = static_cast<size_t>(opt.m) * opt.n * sizeof(float);
    auto run_once = [&]() -> double {
        ACL_CHECK(aclrtMemset(cDevice, cBytes, 0, cBytes));
        const auto t0 = std::chrono::high_resolution_clock::now();
        ACLRT_LAUNCH_KERNEL(static_tensor_matmul_baseline)(1, stream,
            aDevice, bTransposedDevice, cDevice, opt.m, opt.n, opt.k);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        const auto t1 = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(t1 - t0).count();
    };

    for (int i = 0; i < opt.warmup; ++i) (void)run_once();
    std::vector<double> times;
    times.reserve(opt.repeat);
    for (int i = 0; i < opt.repeat; ++i) times.push_back(run_once());

    std::vector<float> out(ref.size());
    ACL_CHECK(aclrtMemcpy(out.data(), cBytes, cDevice, cBytes, ACL_MEMCPY_DEVICE_TO_HOST));

    CaseResult r;
    r.kernel = "static_tensor_matmul_baseline";
    r.path = "static_tensor_matmul_baseline";
    r.m = opt.m; r.n = opt.n; r.k = opt.k;
    r.cores = 1;
    r.baseM = -1; r.baseN = -1; r.baseK = -1;
    r.avgUs = std::accumulate(times.begin(), times.end(), 0.0) / static_cast<double>(times.size());
    r.minUs = *std::min_element(times.begin(), times.end());
    r.gflops = gflops(opt.m, opt.n, opt.k, r.minUs);
    r.readGb = estimate_baseline_read_gb(opt);
    r.writeGb = static_cast<double>(opt.m) * opt.n * sizeof(float) / 1e9;
    check_result(out, ref, r.maxAbs, r.maxRel, r.pass);
    return r;
}



#### 4.2.5 运行单核与多核Matmul API实验方案

`run_tiling`由Host调用两次：

- `matmul_api_tiling_single_core`：请求1核；
- `matmul_api_tiling_multi_core`：请求多核，默认8核。

两次都固定相同`baseM/baseN`，而`baseK`分别由CANN计算。对于相同形状和芯片，两行通常得到相同`baseK`，从而能够较干净地比较单核与多核性能。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
CaseResult run_tiling(const Options &opt,
                      const MatmulLaunchInfo &launchInfo,
                      const char *pathName,
                      uint8_t *aDevice, uint8_t *bDevice, uint8_t *cDevice,
                      uint8_t *workspaceDevice, uint8_t *tilingDevice,
                      aclrtStream stream, const std::vector<float> &ref)
{
    const size_t cBytes = static_cast<size_t>(opt.m) * opt.n * sizeof(float);
    auto run_once = [&]() -> double {
        ACL_CHECK(aclrtMemset(cDevice, cBytes, 0, cBytes));
        const auto t0 = std::chrono::high_resolution_clock::now();
        ACLRT_LAUNCH_KERNEL(static_tensor_matmul_tiling)(launchInfo.blockDim, stream,
            aDevice, bDevice, cDevice, workspaceDevice, tilingDevice);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        const auto t1 = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(t1 - t0).count();
    };

    for (int i = 0; i < opt.warmup; ++i) (void)run_once();
    std::vector<double> times;
    times.reserve(opt.repeat);
    for (int i = 0; i < opt.repeat; ++i) times.push_back(run_once());

    std::vector<float> out(ref.size());
    ACL_CHECK(aclrtMemcpy(out.data(), cBytes, cDevice, cBytes, ACL_MEMCPY_DEVICE_TO_HOST));

    CaseResult r;
    r.kernel = "static_tensor_matmul_tiling";
    r.path = pathName;
    r.m = opt.m; r.n = opt.n; r.k = opt.k;
    r.cores = launchInfo.blockDim;
    r.baseM = launchInfo.baseM;
    r.baseN = launchInfo.baseN;
    r.baseK = launchInfo.baseK;
    r.avgUs = std::accumulate(times.begin(), times.end(), 0.0) / static_cast<double>(times.size());
    r.minUs = *std::min_element(times.begin(), times.end());
    r.gflops = gflops(opt.m, opt.n, opt.k, r.minUs);
    r.readGb = estimate_tiling_read_gb(opt, launchInfo);
    r.writeGb = static_cast<double>(opt.m) * opt.n * sizeof(float) / 1e9;
    check_result(out, ref, r.maxAbs, r.maxRel, r.pass);
    return r;
}



#### 4.2.6 输出表格和CSV

结果表统一包含三行，并保留`baseK`列。需要区分：

- `baseK`列必须保留，因为它展示CANN实际选择；
- 命令行不允许设置`baseK`；
- CSV中的`base_k`同样是输出结果，不是实验输入。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
void print_table(const std::vector<CaseResult> &rows)
{
    std::cout << std::left << std::setw(34) << "kernel" << std::setw(34) << "path"
              << std::setw(7) << "M" << std::setw(7) << "N" << std::setw(7) << "K"
              << std::setw(8) << "cores" << std::setw(8) << "baseM" << std::setw(8) << "baseN"
              << std::setw(8) << "baseK" << std::right << std::setw(12) << "avg_us"
              << std::setw(12) << "min_us" << std::setw(12) << "GFLOPS"
              << std::setw(12) << "read_GB" << std::setw(12) << "write_GB"
              << std::setw(12) << "max_abs" << std::setw(12) << "max_rel"
              << std::setw(8) << "status" << '\n';
    for (const auto &r : rows) {
        std::cout << std::left << std::setw(34) << r.kernel << std::setw(34) << r.path
                  << std::setw(7) << r.m << std::setw(7) << r.n << std::setw(7) << r.k
                  << std::setw(8) << r.cores << std::setw(8) << r.baseM << std::setw(8) << r.baseN
                  << std::setw(8) << r.baseK << std::right << std::fixed << std::setprecision(2)
                  << std::setw(12) << r.avgUs << std::setw(12) << r.minUs << std::setw(12) << r.gflops
                  << std::scientific << std::setprecision(3)
                  << std::setw(12) << r.readGb << std::setw(12) << r.writeGb
                  << std::setw(12) << r.maxAbs << std::setw(12) << r.maxRel
                  << std::fixed << std::setw(8) << (r.pass ? "PASS" : "FAIL") << '\n';
    }
}

void write_csv(const std::string &path, const std::vector<CaseResult> &rows)
{
    std::ofstream out(path);
    if (!out) {
        throw std::runtime_error("failed to open csv: " + path);
    }
    out << "kernel,path,m,n,k,cores,base_m,base_n,base_k,avg_us,min_us,gflops,estimated_read_gb,estimated_write_gb,max_abs,max_rel,status\n";
    for (const auto &r : rows) {
        out << r.kernel << ',' << r.path << ',' << r.m << ',' << r.n << ',' << r.k << ','
            << r.cores << ',' << r.baseM << ',' << r.baseN << ',' << r.baseK << ','
            << std::fixed << std::setprecision(3) << r.avgUs << ',' << r.minUs << ','
            << std::setprecision(6) << r.gflops << ',' << r.readGb << ',' << r.writeGb << ','
            << r.maxAbs << ',' << r.maxRel << ',' << (r.pass ? "PASS" : "FAIL") << '\n';
    }
    if (!out) {
        throw std::runtime_error("failed to write csv: " + path);
    }
}

} // namespace



#### 4.2.7 Host主函数

主函数为同一组FP16输入同时上传行优先B和转置后的$B^T$：单核静态Tensor基线使用$B^T$，单核及多核Matmul API实验方案使用行优先B。随后创建单核、多核两份Matmul Tiling，依次运行三种实验方案，最后输出结果并可靠释放全部Device资源。


In [ ]:
%%writefile -a Source/02.02/ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
int main(int argc, char **argv)
{
    bool aclInited = false;
    int activeDevice = -1;
    aclrtStream stream = nullptr;
    uint8_t *aDevice = nullptr;
    uint8_t *bDevice = nullptr;
    uint8_t *bTransposedDevice = nullptr;
    uint8_t *cDevice = nullptr;
    uint8_t *workspaceDevice = nullptr;
    uint8_t *singleCoreTilingDevice = nullptr;
    uint8_t *multiCoreTilingDevice = nullptr;

    auto cleanup = [&]() {
        if (aDevice != nullptr) {
            (void)aclrtFree(aDevice);
            aDevice = nullptr;
        }
        if (bDevice != nullptr) {
            (void)aclrtFree(bDevice);
            bDevice = nullptr;
        }
        if (bTransposedDevice != nullptr) {
            (void)aclrtFree(bTransposedDevice);
            bTransposedDevice = nullptr;
        }
        if (cDevice != nullptr) {
            (void)aclrtFree(cDevice);
            cDevice = nullptr;
        }
        if (workspaceDevice != nullptr) {
            (void)aclrtFree(workspaceDevice);
            workspaceDevice = nullptr;
        }
        if (singleCoreTilingDevice != nullptr) {
            (void)aclrtFree(singleCoreTilingDevice);
            singleCoreTilingDevice = nullptr;
        }
        if (multiCoreTilingDevice != nullptr) {
            (void)aclrtFree(multiCoreTilingDevice);
            multiCoreTilingDevice = nullptr;
        }
        if (stream != nullptr) {
            (void)aclrtDestroyStream(stream);
            stream = nullptr;
        }
        if (activeDevice >= 0) {
            (void)aclrtResetDevice(activeDevice);
            activeDevice = -1;
        }
        if (aclInited) {
            (void)aclFinalize();
            aclInited = false;
        }
    };

    try {
        Options opt = parse_args(argc, argv);

        stmatmul::MatmulConfig hostCfg;
        hostCfg.m = opt.m;
        hostCfg.n = opt.n;
        hostCfg.k = opt.k;
        hostCfg.seed = opt.seed;
        const auto input = stmatmul::make_input(hostCfg);
        const auto ref = stmatmul::cpu_reference(input, hostCfg);

        const size_t aBytes = static_cast<size_t>(opt.m) * opt.k * sizeof(uint16_t);
        const size_t bBytes = static_cast<size_t>(opt.k) * opt.n * sizeof(uint16_t);
        const size_t cBytes = static_cast<size_t>(opt.m) * opt.n * sizeof(float);

        ACL_CHECK(aclInit(nullptr));
        aclInited = true;
        ACL_CHECK(aclrtSetDevice(opt.device));
        activeDevice = opt.device;
        ACL_CHECK(aclrtCreateStream(&stream));

        const char *runtimeSocName = aclrtGetSocName();
        const auto singleCoreLaunchInfo = make_matmul_launch_info(
            opt, runtimeSocName, opt.baselineCores,
            opt.blockM, opt.blockN);
        const auto multiCoreLaunchInfo = make_matmul_launch_info(
            opt, runtimeSocName, opt.optimizedCores,
            opt.blockM, opt.blockN);
        const size_t workspaceBytes =
            std::max(singleCoreLaunchInfo.workspaceBytes, multiCoreLaunchInfo.workspaceBytes);
        const size_t singleCoreTilingBytes = singleCoreLaunchInfo.tilingBytes.size();
        const size_t multiCoreTilingBytes = multiCoreLaunchInfo.tilingBytes.size();

        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&aDevice), aBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&bDevice), bBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&bTransposedDevice), bBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&cDevice), cBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&workspaceDevice),
                              workspaceBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&singleCoreTilingDevice),
                              singleCoreTilingBytes, ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&multiCoreTilingDevice),
                              multiCoreTilingBytes, ACL_MEM_MALLOC_HUGE_FIRST));

        ACL_CHECK(aclrtMemcpy(aDevice, aBytes, input.a_fp16.data(), aBytes, ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(bDevice, bBytes, input.b_fp16.data(), bBytes, ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(bTransposedDevice, bBytes, input.b_transposed_fp16.data(), bBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(singleCoreTilingDevice, singleCoreTilingBytes,
                              singleCoreLaunchInfo.tilingBytes.data(), singleCoreTilingBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemcpy(multiCoreTilingDevice, multiCoreTilingBytes,
                              multiCoreLaunchInfo.tilingBytes.data(), multiCoreTilingBytes,
                              ACL_MEMCPY_HOST_TO_DEVICE));

        std::vector<CaseResult> rows;
        rows.push_back(run_baseline(opt, aDevice, bTransposedDevice, cDevice, stream, ref));
        rows.push_back(run_tiling(opt, singleCoreLaunchInfo, "matmul_api_tiling_single_core",
                                  aDevice, bDevice, cDevice, workspaceDevice,
                                  singleCoreTilingDevice, stream, ref));
        rows.push_back(run_tiling(opt, multiCoreLaunchInfo, "matmul_api_tiling_multi_core",
                                  aDevice, bDevice, cDevice, workspaceDevice,
                                  multiCoreTilingDevice, stream, ref));

        print_table(rows);
        write_csv(opt.csv, rows);
        std::cout << "[info] csv written to " << opt.csv << '\n';

        const bool allPass = std::all_of(rows.begin(), rows.end(), [](const CaseResult &r) { return r.pass; });

        cleanup();
        return allPass ? 0 : 2;
    } catch (const std::exception &e) {
        cleanup();
        std::cerr << "[error] " << e.what() << '\n';
        return 1;
    }
}


### 4.3 CMake构建配置

工程分别编译两个Device Kernel：

- `static_tensor_matmul_baseline_kernels`：实现单核静态Tensor基线；
- `static_tensor_matmul_tiling_kernels`：实现单核和多核Matmul API实验方案共用的Tiling Kernel。

两个Kernel分别完成静态Tensor基线计算和Matmul API分块计算，Host通过不同的启动配置完成三种实验方案。


In [ ]:
%%writefile Source/02.02/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
project(ascendc_static_tensor_matmul LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C experiment" OFF)

add_library(static_tensor_matmul_host_utils
    Code/half_utils.cpp
    Code/data_utils.cpp
)
target_include_directories(static_tensor_matmul_host_utils PUBLIC Code/include)
target_compile_options(static_tensor_matmul_host_utils PRIVATE
    $<$<CXX_COMPILER_ID:GNU,Clang>:-O3 -Wall -Wextra -Wpedantic>
)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
  set(SOC_VERSION "ascend910b1" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(static_tensor_matmul_baseline_kernels STATIC
      ascend_ops/op_kernel/static_tensor_matmul_baseline.cpp
  )
  ascendc_include_directories(static_tensor_matmul_baseline_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/Code/include
  )
  ascendc_compile_definitions(static_tensor_matmul_baseline_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  ascendc_library(static_tensor_matmul_tiling_kernels STATIC
      ascend_ops/op_kernel/static_tensor_matmul_tiling.cpp
  )
  ascendc_include_directories(static_tensor_matmul_tiling_kernels PRIVATE
      ${CMAKE_CURRENT_SOURCE_DIR}/Code/include
  )
  ascendc_compile_definitions(static_tensor_matmul_tiling_kernels PRIVATE
      -DASCENDC_DUMP=0
      -DHAVE_WORKSPACE
      -DHAVE_TILING
  )

  add_executable(static_tensor_matmul_ascend_demo
      ascend_ops/host_launch/static_tensor_matmul_npu_main.cpp
  )
  target_include_directories(static_tensor_matmul_ascend_demo PRIVATE
      Code/include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/include/external
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/tikcfw
      ${CMAKE_INSTALL_PREFIX}/include/static_tensor_matmul_baseline_kernels
      ${CMAKE_INSTALL_PREFIX}/include/static_tensor_matmul_tiling_kernels
      ${CMAKE_BINARY_DIR}/out/include/static_tensor_matmul_baseline_kernels
      ${CMAKE_BINARY_DIR}/out/include/static_tensor_matmul_tiling_kernels
  )
  target_link_directories(static_tensor_matmul_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
      ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
  )
  target_compile_options(static_tensor_matmul_ascend_demo PRIVATE
      $<$<CXX_COMPILER_ID:GNU,Clang>:-O3 -Wall -Wextra -Wno-deprecated-declarations>
  )
  target_compile_definitions(static_tensor_matmul_ascend_demo PRIVATE
      SOC_VERSION="${SOC_VERSION}"
  )
  target_link_libraries(static_tensor_matmul_ascend_demo PRIVATE
      static_tensor_matmul_baseline_kernels
      static_tensor_matmul_tiling_kernels
      static_tensor_matmul_host_utils
      ascendcl
      tiling_api
      register
      platform
      ascendalog
      c_sec
      dl
  )
  add_dependencies(static_tensor_matmul_ascend_demo
      static_tensor_matmul_baseline_kernels
      static_tensor_matmul_tiling_kernels
  )
endif()

install(DIRECTORY ascend_ops DESTINATION share/ascendc_static_tensor_matmul)


### 4.4 完整构建与运行脚本

`scripts/run.sh`加载CANN环境、清理旧构建、配置CMake并运行实验。脚本提供`-p/-q`设置`baseM/baseN`。


In [ ]:
%%writefile Source/02.02/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "$0")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build_ascend"
RESULT_DIR="${SCRIPT_DIR}/results"
ASCEND_INSTALL_PATH_DEFAULT="/usr/local/Ascend/ascend-toolkit/latest"
if [[ -z "${ASCEND_INSTALL_PATH:-}" || ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  ASCEND_INSTALL_PATH=""
  for candidate in \
    /opt/conda/Ascend/cann-* \
    /opt/Ascend/cann-* \
    /home/ma-user/Ascend/ascend-toolkit/latest \
    "${ASCEND_INSTALL_PATH_DEFAULT}"; do
    if [[ -d "${candidate}" && -f "${candidate}/set_env.sh" ]]; then
      ASCEND_INSTALL_PATH="${candidate}"
      break
    fi
  done
fi
ASCEND_INSTALL_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_INSTALL_PATH_DEFAULT}}"
SOC_VERSION="${SOC_VERSION:-ascend910b1}"
DEVICE_ID=0
BUILD_TYPE="Release"
CLEAN=1
WARMUP=2
REPEAT=3
M=512
N=512
K=512
BLOCK_M=64
BLOCK_N=64
BASELINE_CORES=1
OPTIMIZED_CORES=8
EXTRA_ARGS=()

usage() {
  cat <<USAGE
Usage: bash scripts/run.sh [options] [-- extra_args_for_binary]

Options:
  -a <path>   ASCEND_INSTALL_PATH; optional because common CANN locations are auto-detected
  -v <soc>    SOC_VERSION, default: ascend910b1. Example: ascend910b2, ascend910b3, ascend310p3
  -d <id>     device id, default: 0
  -m <num>    matrix M, default: 512
  -n <num>    matrix N, default: 512
  -k <num>    matrix K, default: 512
  -p <num>    block M/baseM, default: 64
  -q <num>    block N/baseN, default: 64
  -b <num>    single-core case core count, must remain 1
  -o <num>    optimized usedCoreNum, default: 8
  -w <num>    warmup count, default: 2
  -r <num>    repeat count, default: 3
  -t <type>   CMake build type, default: Release
  -c          clean build directory before building; enabled by default for this final Ascend path
  -h          show help

Examples:
  bash scripts/run.sh
  bash scripts/run.sh -a /usr/local/Ascend/ascend-toolkit/latest -v ascend910b1 -d 0
  bash scripts/run.sh -m 1024 -n 1024 -k 1024 -p 64 -q 64 -o 8
USAGE
}

while getopts ":a:v:d:m:n:k:p:q:b:o:w:r:t:ch" opt; do
  case ${opt} in
    a) ASCEND_INSTALL_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    d) DEVICE_ID="${OPTARG}" ;;
    m) M="${OPTARG}" ;;
    n) N="${OPTARG}" ;;
    k) K="${OPTARG}" ;;
    p) BLOCK_M="${OPTARG}" ;;
    q) BLOCK_N="${OPTARG}" ;;
    b) BASELINE_CORES="${OPTARG}" ;;
    o) OPTIMIZED_CORES="${OPTARG}" ;;
    w) WARMUP="${OPTARG}" ;;
    r) REPEAT="${OPTARG}" ;;
    t) BUILD_TYPE="${OPTARG}" ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} requires a value." >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))

if [[ $# -gt 0 && "$1" == "--" ]]; then
  shift
fi
EXTRA_ARGS=("$@")

if [[ ! -d "${ASCEND_INSTALL_PATH}" ]]; then
  echo "ASCEND_INSTALL_PATH does not exist: ${ASCEND_INSTALL_PATH}" >&2
  echo "Use: bash scripts/run.sh -a /path/to/ascend-toolkit/latest" >&2
  exit 1
fi

if [[ -f "${ASCEND_INSTALL_PATH}/set_env.sh" ]]; then
  # shellcheck disable=SC1090
  source "${ASCEND_INSTALL_PATH}/set_env.sh"
fi

export ASCEND_INSTALL_PATH
export ASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
export SOC_VERSION

if [[ "${CLEAN}" == "1" ]]; then
  rm -rf "${BUILD_DIR}"
fi
mkdir -p "${BUILD_DIR}" "${RESULT_DIR}"
cd "${BUILD_DIR}"

cmake "${SCRIPT_DIR}" \
  -DCMAKE_BUILD_TYPE="${BUILD_TYPE}" \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${ASCEND_INSTALL_PATH}" \
  -DASCEND_CANN_PACKAGE_PATH="${ASCEND_INSTALL_PATH}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE=npu

cmake --build . -j

BIN="${BUILD_DIR}/static_tensor_matmul_ascend_demo"
if [[ ! -x "${BIN}" ]]; then
  echo "Cannot find executable: ${BIN}" >&2
  exit 1
fi

CSV="${RESULT_DIR}/static_tensor_matmul_result.csv"
CMD=("${BIN}" --device "${DEVICE_ID}" --m "${M}" --n "${N}" --k "${K}" \
     --block-m "${BLOCK_M}" --block-n "${BLOCK_N}" \
     --baseline-cores "${BASELINE_CORES}" --optimized-cores "${OPTIMIZED_CORES}" \
     --warmup "${WARMUP}" --repeat "${REPEAT}" --csv "${CSV}")
CMD+=("${EXTRA_ARGS[@]}")

echo "[RUN] ${CMD[*]}"
"${CMD[@]}"


In [ ]:
!chmod +x Source/02.02/scripts/run.sh
!find Source/02.02 -maxdepth 3 -type f | sort


### 4.5 运行完整实验

默认命令运行三种实验方案：

1. `static_tensor_matmul_baseline`：单核静态Tensor基线，在LM中完成乘加累加；
2. `matmul_api_tiling_single_core`：单核Matmul API实验方案，请求`baseM=64/baseN=64`；
3. `matmul_api_tiling_multi_core`：多核Matmul API实验方案，默认8核并请求相同`baseM/baseN`。

命令中没有baseK参数。运行完成后，请以表格中实际输出的`baseK`为准。


In [ ]:
!cd Source/02.02 && \
bash scripts/run.sh -m 512 -n 512 -k 512 -p 64 -q 64 -b 1 -o 8 -w 2 -r 3


### 4.6 读取结果并计算两类加速比

结果保存在`Source/02.02/results/static_tensor_matmul_result.csv`。下面计算：

- 单核整体加速：单核静态Tensor基线耗时 ÷ 单核Matmul API实验方案耗时；
- 多核加速：相同`baseM/baseN`下，单核Matmul API实验方案耗时 ÷ 多核Matmul API实验方案耗时。

第一项同时包含Cube、Tiling和数据搬运方式变化，不能解释为纯粹的分块加速；第二项更适合观察多核效果。


In [ ]:
import csv
from pathlib import Path

csv_path = Path("Source/02.02/results/static_tensor_matmul_result.csv")
if not csv_path.exists():
    print("Result CSV does not exist yet:", csv_path)
    print("Please run the Ascend experiment cell first.")
else:
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        print(row)

    by_path = {row["path"]: row for row in rows}
    baseline = by_path.get("static_tensor_matmul_baseline")
    tiled_single = by_path.get("matmul_api_tiling_single_core")
    tiled_multi = by_path.get("matmul_api_tiling_multi_core")
    if baseline and tiled_single:
        baseline_us = float(baseline["avg_us"])
        single_us = float(tiled_single["avg_us"])
        print(f"Single-core speedup (GM-LM baseline -> Matmul API): {baseline_us / single_us:.2f}x")
    if tiled_single and tiled_multi:
        single_us = float(tiled_single["avg_us"])
        multi_us = float(tiled_multi["avg_us"])
        print(f"Multicore speedup (same baseM/baseN): {single_us / multi_us:.2f}x")


### 4.7 扩展实验

可以调整`baseM/baseN`、矩阵规模和多核数量。每次最好只改变一个变量：

- 改变`-p/-q`观察M/N方向分块；
- 改变`-o`观察多核扩展；
- 改变`-m/-n/-k`观察矩阵规模影响。


In [ ]:
experiments = [
    "bash scripts/run.sh -m 512 -n 512 -k 512 -p 32 -q 64 -b 1 -o 8 -w 2 -r 3",
    "bash scripts/run.sh -m 512 -n 512 -k 512 -p 64 -q 32 -b 1 -o 8 -w 2 -r 3",
    "bash scripts/run.sh -m 512 -n 512 -k 512 -p 64 -q 64 -b 1 -o 4 -w 2 -r 3",
    "bash scripts/run.sh -m 1024 -n 1024 -k 1024 -p 64 -q 64 -b 1 -o 8 -w 2 -r 3",
]

for cmd in experiments:
    print("cd Source/02.02 &&", cmd)


---
## 5. 实验总结

本实验实现了三种Ascend NPU实验方案：单核静态Tensor基线、单核Matmul API实验方案和多核Matmul API实验方案。

- 单核静态Tensor基线使用静态`LocalTensor`展示GM→LM、LM内累加和LM→GM写回；
- 单核Matmul API实验方案展示Cube和Tiling带来的整体优化；
- 多核Matmul API实验方案展示相同请求分块下的并行加速；
- 三种实验方案共享输入和CPU参考结果，统一报告误差与性能。
